# Conjectura dos Cortes: Verificação Computacional e Busca de Semiprimos
**Autor da formulação:** Bruno Paiva Viana

Este notebook compila o motor acelerador em C++ (via `pybind11`) para explorar e certificar a Conjectura dos Cortes para valores de $n$ configuráveis.

In [ ]:
# 1. Montar o Google Drive para salvamento persistente
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/ConjecturaCortes'
os.makedirs(DRIVE_DIR, exist_ok=True)

In [ ]:
# 2. Instalar dependências de desenvolvimento do GMP e Pybind11
!apt-get update -qq && apt-get install -y -qq libgmp3-dev libgmpxx4ldbl
!pip install -q pybind11 pandas matplotlib

In [ ]:
# 3. Escrever o código-fonte C++ do motor acelerador
%%writefile conjecture_accelerator.cpp
#include <pybind11/pybind11.h>
#include <pybind11/stl.h>
#include <gmpxx.h>
#include <vector>
#include <cmath>

namespace py = pybind11;

std::vector<unsigned long> get_odd_divisors(unsigned long t) {
    std::vector<unsigned long> divisors;
    unsigned long limit = std::sqrt(t);
    for (unsigned long r = 3; r <= limit; r += 2) {
        if (t % r == 0) {
            divisors.push_back(r);
            unsigned long other = t / r;
            if (other != r && other % 2 != 0) {
                divisors.push_back(other);
            }
        }
    }
    if (t % 2 != 0 && t >= 3) {
        divisors.push_back(t);
    }
    return divisors;
}

bool is_prime_gmp(const mpz_class& N) {
    if (N <= 1) return false;
    if (N == 2 || N == 3) return true;
    if (N % 2 == 0) return false;

    mpz_class temp = N - 1;
    unsigned long m = 0;
    while (temp % 2 == 0) {
        temp /= 2;
        m++;
    }
    mpz_class k = temp;

    mpz_class two_to_m;
    mpz_ui_pow_ui(two_to_m.get_mpz_t(), 2, m);
    if (k < two_to_m) {
        mpz_class exponent = (N - 1) / 2;
        mpz_class result;
        mpz_powm(result.get_mpz_t(), mpz_class(2).get_mpz_t(), exponent.get_mpz_t(), N.get_mpz_t());
        if (result == N - 1) {
            return true;
        }
    }
    return mpz_probab_prime_p(N.get_mpz_t(), 25) > 0;
}

py::dict search_giant_candidate(unsigned long n_target, unsigned long max_s_limit) {
    py::dict result;
    unsigned long limit_s = std::min(2 * n_target - 1, max_s_limit);

    for (unsigned long s = 1; s <= limit_s; ++s) {
        unsigned long t = 2 * n_target - s;
        std::vector<unsigned long> divisors = get_odd_divisors(t);

        for (unsigned long r : divisors) {
            unsigned long u = t / r;
            
            mpz_class A, B;
            mpz_ui_pow_ui(A.get_mpz_t(), 2, u);
            A += 1;

            mpz_ui_pow_ui(B.get_mpz_t(), 2, t);
            B += 1;
            B /= A;

            for (unsigned long a = 1; a < s; ++a) {
                unsigned long b = s - a;

                mpz_class P1, Q1;
                mpz_ui_pow_ui(P1.get_mpz_t(), 2, a);
                P1 *= A;
                P1 += 1;

                mpz_ui_pow_ui(Q1.get_mpz_t(), 2, b);
                Q1 *= B;
                Q1 += 1;

                if (P1 != Q1 && is_prime_gmp(P1) && is_prime_gmp(Q1)) {
                    result["n"] = n_target;
                    result["s"] = s;
                    result["t"] = t;
                    result["r"] = r;
                    result["u"] = u;
                    result["a"] = a;
                    result["b"] = b;
                    result["P"] = P1.get_str();
                    result["Q"] = Q1.get_str();
                    return result;
                }

                mpz_class P2, Q2;
                mpz_ui_pow_ui(P2.get_mpz_t(), 2, a);
                P2 *= B;
                P2 += 1;

                mpz_ui_pow_ui(Q2.get_mpz_t(), 2, b);
                Q2 *= A;
                Q2 += 1;

                if (P2 != Q2 && is_prime_gmp(P2) && is_prime_gmp(Q2)) {
                    result["n"] = n_target;
                    result["s"] = s;
                    result["t"] = t;
                    result["r"] = r;
                    result["u"] = u;
                    result["a"] = a;
                    result["b"] = b;
                    result["P"] = P2.get_str();
                    result["Q"] = Q2.get_str();
                    return result;
                }
            }
        }
    }
    return result;
}

PYBIND11_MODULE(conjecture_accelerator, m) {
    m.doc() = "Motor acelerador para a Conjectura dos Cortes";
    m.def("search_giant_candidate", &search_giant_candidate, "Busca corte bom para dado n");
}

In [ ]:
# 4. Compilar a extensão C++
!g++ -O3 -Wall -shared -std=c++17 -fPIC $(python3 -m pybind11 --includes) conjecture_accelerator.cpp -lgmpxx -lgmp -o conjecture_accelerator.so
print("Módulo compilado com sucesso!")

In [ ]:
# 5. Execução de varredura automatizada com salvamento em loop contínuo
import conjecture_accelerator
import pandas as pd
from datetime import datetime

results_csv = os.path.join(DRIVE_DIR, 'cortes_encontrados_automacao.csv')

start_n = 3
max_n = 150
max_s_limit = 300

print(f"Iniciando varredura automatizada de n = {start_n} até {max_n}...")

for n in range(start_n, max_n + 1):
    res = conjecture_accelerator.search_giant_candidate(n, max_s_limit)
    if res:
        df_row = pd.DataFrame([res])
        print(f"[{datetime.now().strftime('%H:%M:%S')}] n={n} encontrado! s={res['s']}, P (tamanho ~{len(res['P'])} dígitos)")
        
        if not os.path.exists(results_csv):
            df_row.to_csv(results_csv, index=False)
        else:
            df_row.to_csv(results_csv, mode='a', header=False, index=False)
    else:
        print(f"Atenção: Nenhum corte encontrado para n = {n} dentro do limite de s.")